## Fine tuning the Aria model on jazz MIDIs

Create train/val/test splits and create generation examples from test MIDIs.

In [1]:
from jazzgen.dataset_utils import get_midi_paths, create_split, tokenize_midi_paths
from jazzgen.midi_utils import save_first_notes
import os

midi_paths = get_midi_paths()
split = create_split(midi_paths)

train_items = tokenize_midi_paths(
    split["train"],
    out_dir="../aria_tokens/train"
)

val_items = tokenize_midi_paths(
    split["val"],
    out_dir="../aria_tokens/val"
)

test_items = tokenize_midi_paths(
    split["test"],
    out_dir="../aria_tokens/test"
)

# make new midis with fist 10 nodes of test files as other examples of generation
save_dir = os.path.join("..", "data", "test")
os.makedirs(save_dir, exist_ok=True)
for i, item in enumerate(test_items):
    path = os.path.join("..", item["midi_path"])
    save_path = os.path.join(save_dir, f"s-{i:02}.mid")
    save_first_notes(path, save_path, n=10)


Create the datasets.

In [2]:
from jazzgen.dataset_utils import AriaChunkDataset

train_dataset = AriaChunkDataset(
    train_items,
    block_size=2048,
    stride=1024,
)

val_dataset = AriaChunkDataset(
    val_items,
    block_size=2048,
    stride=1024,
)

Load the base model.

In [3]:
import torch
from jazzgen.modeling_utils import load_model

MODEL_ID = "loubb/aria-medium-base"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer, base_model = load_model()

Inspect modules in the model.

In [4]:
for name, module in base_model.named_modules():
    print(name)


model
model.tok_embeddings
model.out_layer_norm
model.encode_layers
model.encode_layers.0
model.encode_layers.0.mixed_qkv
model.encode_layers.0.att_proj_linear
model.encode_layers.0.ff_gate_proj
model.encode_layers.0.ff_up_proj
model.encode_layers.0.ff_down_proj
model.encode_layers.0.norm1
model.encode_layers.0.norm2
model.encode_layers.1
model.encode_layers.1.mixed_qkv
model.encode_layers.1.att_proj_linear
model.encode_layers.1.ff_gate_proj
model.encode_layers.1.ff_up_proj
model.encode_layers.1.ff_down_proj
model.encode_layers.1.norm1
model.encode_layers.1.norm2
model.encode_layers.2
model.encode_layers.2.mixed_qkv
model.encode_layers.2.att_proj_linear
model.encode_layers.2.ff_gate_proj
model.encode_layers.2.ff_up_proj
model.encode_layers.2.ff_down_proj
model.encode_layers.2.norm1
model.encode_layers.2.norm2
model.encode_layers.3
model.encode_layers.3.mixed_qkv
model.encode_layers.3.att_proj_linear
model.encode_layers.3.ff_gate_proj
model.encode_layers.3.ff_up_proj
model.encode_layer

### Fine-tuning only on attention modules

Define LoRA config and model.

In [5]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "mixed_qkv",
        "att_proj_linear",
    ],
    bias="none",
)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 660,897,792 || trainable%: 0.3570


Train the model.

In [6]:
from transformers import TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
import os

output_dir = "../aria_lora_attention"

training_args = TrainingArguments(
    output_dir=output_dir,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=5,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,
    save_steps=250,

    bf16=torch.cuda.is_available(),
    fp16=False,

    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

last_checkpoint = None
if os.path.isdir(output_dir):
    last_checkpoint = get_last_checkpoint(output_dir)

trainer.train(resume_from_checkpoint=last_checkpoint)

model.save_pretrained(f"{output_dir}/final_adapter")

  0%|          | 0/3385 [00:00<?, ?it/s]

{'loss': 2.7234, 'grad_norm': 3.6010818481445312, 'learning_rate': 9.80392156862745e-05, 'epoch': 0.07}
{'loss': 2.6738, 'grad_norm': 3.078329563140869, 'learning_rate': 0.000196078431372549, 'epoch': 0.15}
{'loss': 2.6204, 'grad_norm': 2.6442344188690186, 'learning_rate': 0.00019989452874793578, 'epoch': 0.22}
{'loss': 2.6056, 'grad_norm': 2.515130043029785, 'learning_rate': 0.00019956059820218982, 'epoch': 0.3}
{'loss': 2.6016, 'grad_norm': 2.417072057723999, 'learning_rate': 0.000198998789533872, 'epoch': 0.37}


  0%|          | 0/622 [00:00<?, ?it/s]

{'eval_loss': 2.5972540378570557, 'eval_runtime': 119.9715, 'eval_samples_per_second': 5.185, 'eval_steps_per_second': 5.185, 'epoch': 0.37}
{'loss': 2.618, 'grad_norm': 2.316112518310547, 'learning_rate': 0.00019821038863224866, 'epoch': 0.44}
{'loss': 2.6116, 'grad_norm': 2.318154811859131, 'learning_rate': 0.00019719720001959613, 'epoch': 0.52}
{'loss': 2.6054, 'grad_norm': 2.459757089614868, 'learning_rate': 0.00019596154272094067, 'epoch': 0.59}
{'loss': 2.6119, 'grad_norm': 2.3429999351501465, 'learning_rate': 0.00019450624495618497, 'epoch': 0.66}
{'loss': 2.5833, 'grad_norm': 2.3187506198883057, 'learning_rate': 0.00019283463766676975, 'epoch': 0.74}


  0%|          | 0/622 [00:00<?, ?it/s]

...

{'eval_loss': 2.5752336978912354, 'eval_runtime': 119.8441, 'eval_samples_per_second': 5.19, 'eval_steps_per_second': 5.19, 'epoch': 4.06}
{'loss': 2.5493, 'grad_norm': 2.887146234512329, 'learning_rate': 1.5264001157910656e-05, 'epoch': 4.13}
{'loss': 2.5532, 'grad_norm': 2.9320027828216553, 'learning_rate': 1.2821211381481114e-05, 'epoch': 4.21}
{'loss': 2.5182, 'grad_norm': 3.1830332279205322, 'learning_rate': 1.0577959766103297e-05, 'epoch': 4.28}
{'loss': 2.5183, 'grad_norm': 3.278559446334839, 'learning_rate': 8.539380752266745e-06, 'epoch': 4.35}
{'loss': 2.5399, 'grad_norm': 3.0643365383148193, 'learning_rate': 6.710140317946423e-06, 'epoch': 4.43}


  0%|          | 0/622 [00:00<?, ?it/s]

{'eval_loss': 2.5753159523010254, 'eval_runtime': 120.0595, 'eval_samples_per_second': 5.181, 'eval_steps_per_second': 5.181, 'epoch': 4.43}
{'loss': 2.5268, 'grad_norm': 3.094005584716797, 'learning_rate': 5.094425298933136e-06, 'epoch': 4.5}
{'loss': 2.5113, 'grad_norm': 2.9522855281829834, 'learning_rate': 3.695933805842855e-06, 'epoch': 4.57}
{'loss': 2.5157, 'grad_norm': 3.960588216781616, 'learning_rate': 2.5178667597390293e-06, 'epoch': 4.65}
{'loss': 2.5056, 'grad_norm': 3.1070034503936768, 'learning_rate': 1.5629205657415658e-06, 'epoch': 4.72}
{'loss': 2.5236, 'grad_norm': 2.9789159297943115, 'learning_rate': 8.33280941391068e-07, 'epoch': 4.8}


  0%|          | 0/622 [00:00<?, ?it/s]

{'eval_loss': 2.5751936435699463, 'eval_runtime': 119.9093, 'eval_samples_per_second': 5.187, 'eval_steps_per_second': 5.187, 'epoch': 4.8}
{'loss': 2.5335, 'grad_norm': 2.963444709777832, 'learning_rate': 3.306179138946264e-07, 'epoch': 4.87}
{'loss': 2.5233, 'grad_norm': 2.930354595184326, 'learning_rate': 5.608199770334999e-08, 'epoch': 4.94}
{'train_runtime': 14033.4698, 'train_samples_per_second': 1.932, 'train_steps_per_second': 0.241, 'train_loss': 2.5589227763729574, 'epoch': 4.99}
